In [3]:
import msprime
import pandas as pd

N = 100
ADMIX_TIME = 10
SPLIT_TIME = 200

#------Define demography------#

demography = msprime.Demography()
demography.add_population(name="A", initial_size=N)
demography.add_population(name="B", initial_size=N)
demography.add_population(name="ADMIX", initial_size=N)
demography.add_population(name="ANC", initial_size=N)

# A and B split from a common ancestral population
demography.add_population_split(time=SPLIT_TIME, derived=["A", "B"], ancestral="ANC")

# ADMIX is formed from A and B
# No Bias 
demography.add_admixture(time=ADMIX_TIME, derived="ADMIX", ancestral=["A", "B"], proportions=[0.5, 0.5]) 

demography.sort_events()


#-----Simulate ancestry------#

ts = msprime.sim_ancestry(
    samples={"ADMIX": N},
    demography=demography,
    ploidy=1,
    sequence_length=16569,
    recombination_rate=0,
    random_seed=42,
    record_migrations=True,
)


In [4]:
## Inspecting the migration records

# Note that migration records in msprime are defined in reverse time.

migration_rows = []

for m in ts.migrations():
    migration_rows.append({
        "migration_id": m.id,
        "node": m.node,
        "time": m.time,
        "source_id": m.source,
        "source_name": ts.population(m.source).metadata["name"],
        "dest_id": m.dest,
        "dest_name": ts.population(m.dest).metadata["name"],
    })

migration_df = pd.DataFrame(migration_rows)
migration_df

,migration_id,node,time,source_id,source_name,dest_id,dest_name
0,0,167,10.0,2,ADMIX,0,A
1,1,170,10.0,2,ADMIX,1,B
2,2,183,10.0,2,ADMIX,0,A
3,3,180,10.0,2,ADMIX,0,A
4,4,131,10.0,2,ADMIX,0,A
5,5,66,10.0,2,ADMIX,1,B
6,6,165,10.0,2,ADMIX,1,B
7,7,168,10.0,2,ADMIX,0,A
8,8,48,10.0,2,ADMIX,0,A
9,9,138,10.0,2,ADMIX,0,A


In [5]:
# For each sample identify whether its maternal lineage traces back
# to population A or population B.

tree = ts.first()
sample_to_origin = {}

for sample in ts.samples():
    origin = None
    for m in ts.migrations():
        # Select only migration records from the admixture event.
        if m.time == ADMIX_TIME:
            # Check whether this sample descends from the lineage
            # associated with this admixture event.
            if tree.is_descendant(sample, m.node):
                origin = ts.population(m.dest).metadata["name"]
                break

    sample_to_origin[sample] = origin

df = pd.DataFrame([
    {"sample": sample, "origin": origin}
    for sample, origin in sample_to_origin.items()
])

df

counts_df = (
    df["origin"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("origin")
    .reset_index(name="percent")
)
display(df)
counts_df

,sample,origin
0,0,B
1,1,B
2,2,A
3,3,B
4,4,B
...,...,...
95,95,B
96,96,B
97,97,B
98,98,B


,origin,percent
0,A,71.0
1,B,29.0
